In [1]:
!pip install jaxlib
!pip install jax
!pip install git+https://github.com/deepmind/dm-haiku
!pip install gym==0.25
!pip install gym[box2d]
!pip install optax
!pip install matplotlib
!pip install chex
!pip install gym[classic_control]

  Cloning https://github.com/deepmind/dm-haiku to /tmp/pip-req-build-zpv0gu03
  Running command git clone --filter=blob:none --quiet https://github.com/deepmind/dm-haiku /tmp/pip-req-build-zpv0gu03
  Resolved https://github.com/deepmind/dm-haiku to commit 0d53d52d82a7a18c805a01596b5058d8a9ea2f93
  Preparing metadata (setup.py) ... done
  Created wheel for dm-haiku: filename=dm_haiku-0.0.17.dev0-py3-none-any.whl size=374766 sha256=2bc6530e9c43a6a80f7c2190fdce6e3a3e00d376fbab3ab3c659a3b251968698
  Stored in directory: /tmp/pip-ephem-wheel-cache-58uipr21/wheels/9e/61/4f/80a3533070997bbf330ff52c178cc780d5e60ad222899e2924
Successfully built dm-haiku
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.4/720.4 kB 11.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for gym: filename=gym-0.25.0-py3-none-any.whl size=824514 sha256=16dfff11dd7c28d8e048c73d15fec959614daac07afe690c30

In [2]:
import copy
from shutil import rmtree # deleting directories
import random
import collections # useful data structures
import numpy as np
import gym # reinforcement learning environments
from gym.wrappers import RecordVideo
import jax
import jax.numpy as jnp # jax numpy
import haiku as hk # jax neural network library
import optax # jax optimizer library
import matplotlib.pyplot as plt # graph plotting library
from IPython.display import HTML
from base64 import b64encode
import chex

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

1

In [3]:
import jax.numpy as jnp
from jax import lax

def policy(params, observation):

    # Вычисляем скалярное произведение векторов
    dot_product = jnp.dot(observation, params)

    # Используем jax.lax.select для условного выбора действия
    # Если dot_product > 0, возвращаем 1, иначе 0
    action = lax.select(
        dot_product > 0,  # условие
        1,                # значение если True
        0                 # значение если False
    )

    return action

# Пример использования:
params = jnp.array([0.5, -0.2, 0.1, 0.3])
observation = jnp.array([0.1, -0.5, 0.2, 0.4])
action = policy(params, observation)
print(f"Action: {action}")  # Выведет действие (0 или 1)

Action: 1


2

In [4]:
import jax.numpy as jnp
import numpy as np

def run_episode(env):
    episode_return = 0
    done = False
    params = jnp.array([1, -2, 2, -1])

    # Получаем начальное наблюдение из среды
    obs = env.reset()
    obs = jnp.array(obs)  # конвертируем в JAX массив

    while not done:
        # Вычисляем действие с помощью линейной политики
        action = policy(params, obs)
        action = np.array(action)

        # Делаем шаг в среде
        obs, reward, done, info = env.step(action)

        # Конвертируем наблюдение обратно в JAX массив для следующего шага
        obs = jnp.array(obs)

        # Добавляем вознаграждение к общему возврату эпизода
        episode_return += reward

    return episode_return

3

In [5]:
def random_policy_search_choose_action(
    key,
    params,
    actor_state,
    obs,
    evaluation=False
):
    # Вычисляем действие, используя лучшие параметры
    best_action = linear_policy(params.best_params, obs)

    # Вычисляем действие, используя текущие параметры
    current_action = linear_policy(params.current_params, obs)

    action = jax.lax.select(
        evaluation,      # условие: если evaluation равно True
        best_action,     # значение, если условие True
        current_action   # значение, если условие False
    )

    # Возвращаем выбранное действие и неизмененный actor_state
    return action, actor_state

4

In [6]:
import jax.numpy as jnp
from jax import random

def get_new_random_weights(random_key, old_weights, minval=-2.0, maxval=2.0):
    new_weights_shape = old_weights.shape
    new_weights_dtype = old_weights.dtype

    # Генерируем случайные веса из равномерного распределения в интервале [minval, maxval]
    new_params = random.uniform(
        key=random_key,
        shape=new_weights_shape,
        dtype=new_weights_dtype,
        minval=minval,
        maxval=maxval
    )

    return new_params

5

In [7]:
def random_policy_search_learn(key, params, learn_state, memory):
    best_params = params.best
    current_params = params.current

    current_average_episode_return = memory
    best_average_episode_return = learn_state.best_average_episode_return

    # Выбираем лучшие параметры
    best_params = jax.lax.select(
        current_average_episode_return > best_average_episode_return,  # условие
        current_params,  # значение, если условие True (текущие параметры лучше)
        best_params      # значение, если условие False (оставляем старые лучшие)
    )

    # Выбираем лучшее среднее значение возврата за эпизод
    best_average_episode_return = jax.lax.select(
        current_average_episode_return > best_average_episode_return,  # условие
        current_average_episode_return,  # значение, если условие True (текущее лучше)
        best_average_episode_return       # значение, если условие False (оставляем старое)
    )

    new_params = get_new_random_weights(key, current_params)

    params = RandomPolicySearchParams(current=new_params, best=best_params)

    return params, RandomPolicyLearnState(best_average_episode_return)

6

In [8]:
import jax.numpy as jnp

def compute_weighted_log_prob(action_prob, episode_return):
    # Вычисляем логарифм вероятности действия
    log_prob = jnp.log(action_prob)

    # Умножаем логарифм вероятности на отдачу эпизода
    weighted_log_prob = log_prob * episode_return

    return weighted_log_prob

7

In [ ]:
def compute_rewards_to_go(rewards):

    rewards_to_go = []

    # Итерация с вычислением суммы от текущего индекса до конца
    for i in range(len(rewards)):
        # Суммируем все награды от текущего шага i до конца
        reward_sum = sum(rewards[i:])
        rewards_to_go.append(reward_sum)

    return rewards_to_go

8

In [9]:
import jax.numpy as jnp
from jax import random

def sample_action(random_key, logits):
    # Преобразуем логиты в вероятности с помощью softmax
    probabilities = jnp.exp(logits - jnp.max(logits)) / jnp.sum(jnp.exp(logits - jnp.max(logits)))

    # Выбираем действие из категориального распределения
    action = random.categorical(key=random_key, logits=logits)

    return action

9

In [10]:
import jax.numpy as jnp
from jax import nn

def policy_gradient_loss(action, logits, reward_to_go):
    # Преобразуем логиты в вероятности с помощью softmax
    all_action_probs = nn.softmax(logits)

    # Извлекаем вероятность конкретного действия, которое было выбрано
    action_prob = all_action_probs[action]

    # Вычисляем взвешенную логарифмическую вероятность
    weighted_log_prob = compute_weighted_log_prob(action_prob, reward_to_go)

    loss = -weighted_log_prob  # отрицательное значение, так как мы хотим градиентный подъем

    return loss

10

In [11]:
import jax.numpy as jnp

def select_greedy_action(q_values):
    # Находим индекс действия с наибольшим Q-значением
    action = jnp.argmax(q_values)

    return action

11

In [12]:
import jax.numpy as jnp

def compute_squared_error(pred, target):
    # Вычисляем квадрат разности между предсказанием и целевым значением
    squared_error = jnp.square(pred - target)

    return squared_error

12

In [13]:
import jax.numpy as jnp

def compute_bellman_target(reward, done, next_q_values):

    # Находим максимальное Q-значение для следующего состояния
    max_next_q = jnp.max(next_q_values)

    # Вычисляем цель Беллмана
    # Если done == 1.0 (терминальное состояние), используем только reward
    # Если done == 0.0 (не терминальное), используем reward + max_next_q
    bellman_target = reward + (1.0 - done) * max_next_q

    return bellman_target

13

In [14]:
def q_learning_loss(q_values, action, reward, done, next_q_values):

    # Извлекаем Q-значение выбранного действия
    chosen_action_q_value = q_values[action]

    # Вычисляем цель Беллмана
    bellman_target = compute_bellman_target(reward, done, next_q_values)

    # Вычисляем квадратичную ошибку между предсказанием и целью
    squared_error = compute_squared_error(chosen_action_q_value, bellman_target)

    return squared_error

14

In [15]:
import jax.random as random

def select_random_action(key, num_actions):
    # Генерируем случайное целое число в диапазоне [0, num_actions)
    action = random.randint(key=key, shape=(), minval=0, maxval=num_actions)

    return action

15

In [16]:
EPSILON_START = 1.0
EPSILON_DECAY_TIMESTEPS = 3000
EPSILON_MIN = 0.1

def get_epsilon(num_timesteps):
    # Линейное затухание эпсилон от EPSILON_START до EPSILON_MIN
    # Чем больше num_timesteps, тем меньше эпсилон
    epsilon = EPSILON_START - (EPSILON_START - EPSILON_MIN) * (num_timesteps / EPSILON_DECAY_TIMESTEPS)

    # Ограничиваем значение эпсилон снизу значением EPSILON_MIN
    epsilon = jax.lax.select(
        epsilon < EPSILON_MIN,
        EPSILON_MIN,  # если меньше минимума, устанавливаем в минимум
        epsilon       # иначе оставляем без изменений
    )

    return epsilon

16

In [17]:
def select_epsilon_greedy_action(key, q_values, num_timesteps):
    num_actions = len(q_values)

    # Получаем текущее значение эпсилон на основе количества временных шагов
    epsilon = get_epsilon(num_timesteps)

    # Генерируем случайное число в диапазоне [0, 1)
    random_number = jax.random.uniform(key)

    # Проверяем, нужно ли исследовать (случайное число меньше эпсилон)
    should_explore = random_number < epsilon

    # Выбираем действие: исследование (случайное) или эксплуатация (жадное)
    action = jax.lax.select(
        should_explore,
        jax.random.randint(key, shape=(), minval=0, maxval=num_actions),  # случайное действие
        jnp.argmax(q_values)  # жадное действие
    )

    return action